In [14]:
# https://github.com/ubisoft/ubisoft-laforge-toxbuster?tab=readme-ov-file
# https://www.ubisoft.com/en-us/studio/laforge/news/54QiUaTH5oCX7MUhSSgrGy/large-language-models-for-toxicity-detection-toxbuster

# A faire la première fois
# import nltk
# nltk.download('punkt_tab')

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn import tree
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
import re

from Models import model_choice

main_path = "C:/Users/oscar/Desktop/ToxBuster"
df = pd.read_csv(main_path + "/CONDA.csv", sep=None, engine="python")

In [16]:
def bloc(df):
    # Création de blocs de chat
    df_bloc = df.sort_values(["matchid", "line_index"])
    cols = ["matchid", "profileid", "end_string_index"]
    new_bloc = (df_bloc[cols] != df_bloc[cols].shift()).any(axis=1)
    df_bloc["chat_bloc_id"] = new_bloc.cumsum()

    # Puisque les annotations de catégorie de toxicité se font par bloc, on les fusionne pour les considérer comme une unique observation
    df_agg = df_bloc.groupby("chat_bloc_id").agg(
        matchid=("matchid", "first"),
        profileid=("profileid", "first"),
        team=("team", "first"),
        full_text=("full_text", lambda s: " | ".join(s.dropna())),
        n_messages=("full_text", "size"),
        start_timestamp=("relative_timestamp", "first"),
        end_timestamp=("relative_timestamp", "last"),
        line_index=("line_index", "first"),
        category_id=("category_id", "min"),
    ).reset_index()
    return df_agg

df = bloc(df)

In [17]:
# Pour clean les ponctuations, chiffres, emojis, etc
# On ne l'applique pas car on pourrait perdre de l'information pour les n-grammes
# def clean_text(text):
#     return re.sub('[^a-zA-Z]', ' ', text)

df = df[~df["full_text"].isnull()]

# df["full_text"] = df["full_text"].apply(clean_text) jsp si je garde
df["tokens"] = df["full_text"].apply(word_tokenize)


df = df[df["tokens"].str.len() > 0]

In [18]:
SEED = 42
accuracy_cross_val = []
precision_cross_val = []
recall_cross_val = []
F1_cross_val = []
match = df["matchid"].unique()
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (match_train, match_test) in enumerate(kf.split(match)):
    df_train = df[df["matchid"].isin(match_train)]
    df_test = df[df["matchid"].isin(match_test)]

    # vectorizer = TfidfVectorizer()
    # X = vectorizer.fit_transform(df_test["full_text"]) jsp ce que c'est

    ### NLP
    list = ["Logistic", "XGBoost"] # ["Logistic", "Multinomial", "RandomForest", "GradientBoosting", "XGBoost"]
    for m in list:
        clf = model_choice(m)
        model = Pipeline([
            ("features", FeatureUnion([
                ("words", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
                ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3)),
            ])),
            ("classifier", clf)
        ])
        model.fit(df_train["full_text"], df_train["category_id"])
        prediction = model.predict(df_test["full_text"])
        Precision = []
        Recall = []
        F1 = []
        for j in range(3):
            TP = ((df_test["category_id"] == j) & (prediction == j)).sum()
            FP = ((df_test["category_id"] != j) & (prediction == j)).sum()
            TN = ((df_test["category_id"] != j) & (prediction != j)).sum()
            FN = ((df_test["category_id"] == j) & (prediction != j)).sum()
            Precision.append(TP / (TP + FP))
            Recall.append(TP / (TP + FN))
            F1.append(2*TP / (2*TP + FP + FN))
        Accuracy = (df_test["category_id"] == prediction).mean()
        Precision_macro = sum(Precision)/3
        Recall_macro = sum(Recall)/3
        F1_macro = sum(F1)/3
        accuracy_cross_val.append(Accuracy)
        precision_cross_val.append(Precision_macro)
        recall_cross_val.append(Recall_macro)
        F1_cross_val.append(F1_macro)
    print(f"||========= {clf.__class__.__name__} =========||")
    print(f"Accuracy: {Accuracy}")
    print(f"Precision: {Precision_macro}")
    print(f"Recall: {Recall_macro}")
    print(f"F1: {F1_macro}")

||========= LogisticRegression =========||
Accuracy: 0.921595598349381
Precision: 0.7966138849251415
Recall: 0.8593352940138872
F1: 0.8245709151879383
||========= XGBClassifier =========||
Accuracy: 0.9423429619440623
Precision: 0.9115724360407453
Recall: 0.8166971998192357
F1: 0.8570068533330456
||========= LogisticRegression =========||
Accuracy: 0.9199496106275767
Precision: 0.811736445445919
Recall: 0.8603350889043465
F1: 0.8336856784397805
||========= XGBClassifier =========||
Accuracy: 0.9352954649564819
Precision: 0.9081489853927146
Recall: 0.8113564673456638
F1: 0.852953019326641
||========= LogisticRegression =========||
Accuracy: 0.9229357798165138
Precision: 0.8001750641614956
Recall: 0.8680333307456302
F1: 0.8300956996510988
||========= XGBClassifier =========||
Accuracy: 0.9443807339449541
Precision: 0.9179714855875766
Recall: 0.823683979217592
F1: 0.8644423717160682
||========= LogisticRegression =========||
Accuracy: 0.9206365609176398
Precision: 0.8076599815066702
Recal